# Mini "LLM" educativo

Uma única camada de pesos, sem embedding, sem camada escondida. Só `numpy`.

Objetivo: ver na prática o que é peso, o que é o cálculo de uma camada, e como o treino ajusta os pesos automaticamente.

## 1) Vocabulário

O modelo só conhece estas palavras. Cada uma vira um número (id).

In [ ]:
import numpy as np

np.random.seed(0)

vocab = ["o", "gato", "correu", "comeu", "peixe", "feliz"]
palavra_para_id = {p: i for i, p in enumerate(vocab)}
V = len(vocab)  # quantidade de palavras que o modelo conhece
V

## 2) Palavra vira vetor (one-hot)

Forma mais simples de representar palavra como número: um vetor de zeros com um único `1` na posição dela.

Ex: `gato` (id=1) vira `[0, 1, 0, 0, 0, 0]`.

In [ ]:
def one_hot(id_palavra):
    v = np.zeros(V)
    v[id_palavra] = 1.0
    return v

one_hot(palavra_para_id["gato"])

## 3) Dados de treino

Frases de exemplo, quebradas em pares `(palavra atual -> próxima palavra)`.

In [ ]:
frases = [
    ["o", "gato", "correu"],
    ["o", "gato", "comeu", "peixe"],
    ["o", "gato", "feliz"],
    ["o", "peixe", "correu"],
]

pares = []
for frase in frases:
    for i in range(len(frase) - 1):
        pares.append((palavra_para_id[frase[i]], palavra_para_id[frase[i + 1]]))

pares

## 4) Os pesos

`W` é uma tabela V x V: uma linha de pesos para cada palavra de entrada, uma coluna para cada palavra de saída possível.

Começa **aleatório**, sem sentido nenhum — é isso que o treino vai corrigir.

In [ ]:
W = np.random.randn(V, V) * 0.1
b = np.zeros(V)  # um ajuste extra por palavra de saída
W

## 5) O cálculo do modelo (forward)

Duas linhas só:

1. `x @ W + b` — o "multiplica e soma" manual: vetor da palavra vezes os pesos, gera uma pontuação bruta por palavra do vocabulário.
2. `softmax` — transforma a pontuação em probabilidade (soma 100%).

In [ ]:
def softmax(pontuacoes):
    e = np.exp(pontuacoes - np.max(pontuacoes))
    return e / e.sum()


def prever(id_entrada):
    x = one_hot(id_entrada)
    pontuacoes = x @ W + b
    probs = softmax(pontuacoes)
    return probs


# antes de treinar, os pesos são aleatórios: previsão é lixo
prever(palavra_para_id["gato"])

## 6) Treino

A cada rodada:
- calcula o erro (quanto menor a probabilidade dada à palavra certa, maior o erro)
- calcula o gradiente: `erro_por_palavra = probs - gabarito` — isso já É a direção de ajuste de cada peso
- atualiza: `peso_novo = peso_velho - passo * gradiente`

In [ ]:
taxa_aprendizado = 0.5
epocas = 1000

for epoca in range(epocas):
    erro_total = 0.0

    for id_entrada, id_alvo in pares:
        x = one_hot(id_entrada)
        probs = prever(id_entrada)

        erro_total += -np.log(probs[id_alvo] + 1e-9)

        gabarito = one_hot(id_alvo)
        erro_por_palavra = probs - gabarito

        gradiente_W = np.outer(x, erro_por_palavra)
        gradiente_b = erro_por_palavra

        W -= taxa_aprendizado * gradiente_W
        b -= taxa_aprendizado * gradiente_b

    if epoca % 200 == 0:
        print(f"época {epoca:4d} | erro médio = {erro_total / len(pares):.4f}")

## 7) Teste

Agora os pesos não são mais aleatórios — veja se o modelo aprendeu o padrão das frases de treino.

In [ ]:
for palavra_teste in ["o", "gato", "peixe"]:
    probs = prever(palavra_para_id[palavra_teste])
    ranking = sorted(zip(vocab, probs), key=lambda t: -t[1])
    top = ranking[0]
    print(f"depois de '{palavra_teste}' -> '{top[0]}' ({top[1]*100:.1f}%)")

## Pra experimentar

Mude a lista `frases` na célula 3, rode as células de novo (a partir da célula dos pesos `W`) e veja a previsão final mudar de acordo com o novo padrão que você ensinar.